# K-Prototypes Clustering on Retail Customer Data

**Objective:** Cluster customers using **both categorical and numerical** retail features with K-Prototypes.

**Why K-Prototypes?** Real retail data mixes product categories (categorical) with income and spending (numerical). K-Prototypes extends K-Modes to handle both.

**Pipeline:** Load Data → EDA → Feature Prep → Cost Analysis → K-Prototypes → Cluster Profiling

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler

from retail_data import generate_retail_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Load Retail Dataset

In [ ]:
df = generate_retail_dataset(n_samples=2000, random_state=RANDOM_STATE)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
cat_cols = ['Region', 'Product_Category', 'Purchase_Channel']
num_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes[0], cat_cols):
    sns.countplot(data=df, x=col, ax=ax, palette='muted')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
for ax, col in zip(axes[1], num_cols[:3]):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Annual_Income', y='Spending_Score', hue='Product_Category', style='Purchase_Channel', alpha=0.6)
plt.title('Income vs Spending by Category & Channel')
plt.show()

## 3. Prepare Mixed-Type Features

K-Prototypes expects a single array with categorical columns first, then numerical. We pass `categorical` indices to indicate which columns are categorical.

In [ ]:
feature_cols = cat_cols + num_cols
df_model = df[feature_cols].copy()

# Scale numerical columns
scaler = StandardScaler()
df_model[num_cols] = scaler.fit_transform(df_model[num_cols])

# Convert to mixed-type array: strings for categorical, floats for numerical
X_mixed = df_model.values
cat_indices = list(range(len(cat_cols)))

print(f'Feature order: {feature_cols}')
print(f'Categorical indices: {cat_indices}')
print(f'Matrix shape: {X_mixed.shape}')

## 4. Find Optimal K (Cost Function)

In [ ]:
k_range = range(2, 9)
costs = []

for k in k_range:
    kp = KPrototypes(n_clusters=k, init='Huang', n_init=3, random_state=RANDOM_STATE, verbose=0)
    kp.fit(X_mixed, categorical=cat_indices)
    costs.append(kp.cost_)
    print(f'K={k}, Cost={kp.cost_:.2f}')

plt.figure(figsize=(10, 5))
plt.plot(k_range, costs, 'mo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Cost')
plt.title('K-Prototypes Cost Function')
plt.show()

## 5. Train Final K-Prototypes Model

In [ ]:
OPTIMAL_K = 4
kproto = KPrototypes(n_clusters=OPTIMAL_K, init='Huang', n_init=10, random_state=RANDOM_STATE, verbose=0)
df['Cluster'] = kproto.fit_predict(X_mixed, categorical=cat_indices)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())
print(f'\nFinal Cost: {kproto.cost_:.2f}')

In [ ]:
centroids = kproto.cluster_centroids_
cat_centroids = centroids[0]
num_centroids = centroids[1]

centroids_df = pd.DataFrame(cat_centroids, columns=cat_cols)
centroids_df[num_cols] = scaler.inverse_transform(num_centroids).round(2)
centroids_df.index.name = 'Cluster'
print('Cluster Centroids:')
centroids_df

## 6. Visualize Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='Annual_Income', y='Spending_Score', hue='Cluster', palette='Set2', ax=axes[0], s=60, alpha=0.7)
axes[0].set_title('Clusters: Income vs Spending')
sns.scatterplot(data=df, x='Num_Purchases', y='Total_Sales', hue='Cluster', palette='Set2', ax=axes[1], s=60, alpha=0.7)
axes[1].set_title('Clusters: Purchases vs Sales')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(df['Cluster'], df[col], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='Pastel1')
    ax.set_title(f'{col} Composition')
    ax.legend(title=col, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 7. Cluster Profiling

In [ ]:
profile = df.groupby('Cluster').agg({
    'Annual_Income': 'mean',
    'Spending_Score': 'mean',
    'Num_Purchases': 'mean',
    'Total_Sales': 'mean',
    'Region': lambda x: x.mode()[0],
    'Product_Category': lambda x: x.mode()[0],
    'Purchase_Channel': lambda x: x.mode()[0],
}).round(2)
profile.columns = ['Avg_Income', 'Avg_Spending', 'Avg_Purchases', 'Avg_Total_Sales', 'Top_Region', 'Top_Category', 'Top_Channel']
profile

In [ ]:
pd.crosstab(df['Cluster'], df['Customer_Segment'], normalize='index').round(3)

## 8. Conclusions

- **K-Prototypes** is the best clustering choice when retail data has both categorical preferences and numerical spending metrics.
- Cluster centroids give interpretable modes (categories) and means (numerical values).
- Use profiles to create omnichannel strategies per cluster (e.g., Online + Electronics + High Income).
- **Limitation:** Slower than K-Means; tune `gamma` parameter if categorical/numerical features are imbalanced in influence.